<a href="https://colab.research.google.com/github/OJB-Quantum/Notebooks-for-Ideas/blob/main/DXF_to_SVG_Converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Authored by Onri Jay Benally (2026)

Open Access (CC-BY-4.0)

In [10]:
# Install the UV package manager
!pip install uv

# Utilize UV to rapidly provision the environment
!uv pip install --system cupy-cuda12x ezdxf gdstk matplotlib

Using Python 3.12.13 environment at: /usr
Checked 4 packages in 44ms


In [9]:
"""
Deep-Scanning Layout Flattening and SVG Export (Color & Background Controlled).

This module processes DXF or OASIS design files. It features configurable
backgrounds (transparent/white) and geometric color assignment. It extracts
geometry utilizing a 3-Tier scan (Model Space, Paper Space, Block Libraries)
and renders the layout inline within the notebook.
"""

import os
import io

# Scientific and display libraries
import cupy as cp
import matplotlib as mpl
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display, HTML

# Layout parsing libraries
import ezdxf
from ezdxf import bbox as ezdxf_bbox
from ezdxf.addons.drawing import Frontend, RenderContext
from ezdxf.addons.drawing.svg import SVGBackend
from ezdxf.addons.drawing import layout as ezdxf_layout
from ezdxf.addons.drawing.config import Configuration, BackgroundPolicy, ColorPolicy
import gdstk

# =========================================================================
# CONTROL KNOBS
# =========================================================================
DEFAULT_SVG_WIDTH_PX: int = 3000
ENABLE_GPU_WARMUP: bool = True
DPI_SETTING: int = 250

# --- Visual Render Controls ---
USE_TRANSPARENT_BACKGROUND: bool = True
ALTERNATIVE_BACKGROUND_COLOR: str = "#FFFFFF"  # Applies if transparent is False

KEEP_ORIGINAL_COLORS: bool = False
FORCED_GEOMETRY_COLOR: str = "#000000"  # Defaults to Black if colors are not kept

# --- Flattening Parameters ---
TARGET_DXF_LAYER: str = "0"
TARGET_OASIS_LAYER: int = 0
OUTPUT_SVG_FILENAME: str = "flattened_layout.svg"

# Apply the global DPI constraint for any matplotlib rendering operations
mpl.rcParams['figure.dpi'] = DPI_SETTING

# =========================================================================
# IMPLEMENTATION
# =========================================================================

def warmup_gpu() -> None:
    """
    Initializes the Compute Unified Device Architecture (CUDA) context for
    CuPy to ensure rapid dispatching of subsequent array operations.
    """
    print("Initializing GPU context via CuPy...")
    try:
        dummy_array = cp.ones((1024, 1024), dtype=cp.float32)
        cp.fft.fft2(dummy_array)
        print("GPU context initialized successfully.")
    except Exception as error:
        print(f"GPU initialization skipped or failed: {error}")


def process_dxf_to_svg(file_path: str, width: int, output_filename: str) -> str:
    """
    Parses a DXF file dynamically via disk. Utilizes a 3-tier scan to extract
    geometry. Applies user-defined color and background configuration policies
    before rendering the SVG string.
    """
    doc = ezdxf.readfile(file_path)

    # ---------------------------------------------------------------------
    # 1. 3-TIER DYNAMIC SPACE SCANNER
    # ---------------------------------------------------------------------
    geometry_space = doc.modelspace()
    entity_count = len(geometry_space)

    if entity_count == 0:
        print("Model Space is empty. Scanning Paper Space layouts...")
        max_entities = 0
        populated_space = None

        for layout in doc.layouts:
            if layout.name.lower() == 'model':
                continue
            if len(layout) > max_entities:
                max_entities = len(layout)
                populated_space = layout

        if populated_space is not None and max_entities > 0:
            geometry_space = populated_space
            print(f"Geometry located! Targeting Paper Layout: '{populated_space.name}' ({max_entities} entities).")
        else:
            print("Paper Spaces are empty. Scanning Block Library for orphaned EDA components...")
            max_block_entities = 0
            populated_block = None

            for block in doc.blocks:
                if block.name.startswith('*'):
                    continue
                if len(block) > max_block_entities:
                    max_block_entities = len(block)
                    populated_block = block

            if populated_block is not None and max_block_entities > 0:
                geometry_space = populated_block
                print(f"Geometry located! Targeting Orphaned Block: '{populated_block.name}' ({max_block_entities} entities).")
            else:
                raise ValueError("All spaces are completely devoid of geometric entities. File may be corrupt.")
    else:
        print(f"Targeting Model Space directly. Found {entity_count} root entities.")

    # ---------------------------------------------------------------------
    # 2. HIERARCHY AND LAYER MANAGEMENT
    # ---------------------------------------------------------------------
    print("Flattening DXF block hierarchy (exploding nested inserts)...")
    inserts_remain = True
    iterations = 0

    while inserts_remain and iterations < 15:
        inserts_remain = False
        for entity in list(geometry_space.query('INSERT')):
            try:
                entity.explode()
                inserts_remain = True
            except Exception:
                pass
        iterations += 1

    if not KEEP_ORIGINAL_COLORS:
        print(f"Flattening geometry to Target Layer: '{TARGET_DXF_LAYER}'...")
        for entity in geometry_space:
            if hasattr(entity.dxf, 'layer'):
                entity.dxf.layer = TARGET_DXF_LAYER
    else:
        print("Preserving native DXF layer assignments to maintain original colors.")

    # ---------------------------------------------------------------------
    # 3. CONFIGURE RENDER ENGINE & SECURE EXTENTS
    # ---------------------------------------------------------------------
    extents = ezdxf_bbox.extents(geometry_space)
    if not extents.has_data:
        print("Warning: Native geometric bounds are empty. Forcing boundary via CAD header extents.")
        min_pt = doc.header.get('$EXTMIN', (0, 0, 0))
        max_pt = doc.header.get('$EXTMAX', (width, width, 0))
        if min_pt == max_pt:
            min_pt, max_pt = (0, 0, 0), (width, width, 0)
        geometry_space.add_line(min_pt, max_pt, dxfattribs={'layer': TARGET_DXF_LAYER})

    render_context = RenderContext(doc)

    # Apply Visual Control Knobs to ezdxf Configuration
    bg_policy = BackgroundPolicy.OFF if USE_TRANSPARENT_BACKGROUND else BackgroundPolicy.CUSTOM
    color_policy = ColorPolicy.COLOR if KEEP_ORIGINAL_COLORS else ColorPolicy.CUSTOM

    config = Configuration(
        background_policy=bg_policy,
        custom_bg_color=ALTERNATIVE_BACKGROUND_COLOR,
        color_policy=color_policy,
        custom_fg_color=FORCED_GEOMETRY_COLOR
    )

    backend = SVGBackend()
    frontend = Frontend(render_context, backend, config=config)

    frontend.draw_layout(geometry_space)

    page_config = ezdxf_layout.Page(
        width=width,
        height=width,
        units=ezdxf_layout.Units.px,
        margins=ezdxf_layout.Margins.all(0)
    )

    svg_string = backend.get_string(page_config)

    with open(output_filename, "w", encoding="utf-8") as file_handle:
        file_handle.write(svg_string)

    return svg_string


def process_oasis_to_svg(file_path: str, width: int, output_filename: str) -> str:
    """
    Parses an OASIS file, flattens the cell hierarchy, applies background/color
    policies, and renders directly to a downloadable SVG file.
    """
    library = gdstk.read_oas(file_path)
    top_cells = library.top_level()

    if not top_cells:
        raise ValueError("No top-level cells found in the OASIS file.")

    main_cell = top_cells[0]

    print("Flattening OASIS cell hierarchy...")
    main_cell.flatten()

    if not KEEP_ORIGINAL_COLORS:
        print(f"Flattening OASIS geometric layers to Target Layer: {TARGET_OASIS_LAYER}...")
        for polygon in main_cell.polygons:
            polygon.layer = TARGET_OASIS_LAYER
        for path in main_cell.paths:
            path.layer = TARGET_OASIS_LAYER
        for label in main_cell.labels:
            label.layer = TARGET_OASIS_LAYER
    else:
        print("Preserving native OASIS layer assignments to maintain original colors.")

    cell_bbox = main_cell.bounding_box()
    if cell_bbox is None:
        raise ValueError("The primary cell bounding box is empty after flattening.")

    physical_width = cell_bbox[1][0] - cell_bbox[0][0]
    scaling_factor = width / physical_width if physical_width > 0 else 1.0

    # Apply Visual Control Knobs to gdstk output
    bg_color = "none" if USE_TRANSPARENT_BACKGROUND else ALTERNATIVE_BACKGROUND_COLOR
    style_spec = {TARGET_OASIS_LAYER: {"fill": FORCED_GEOMETRY_COLOR, "stroke": FORCED_GEOMETRY_COLOR}} if not KEEP_ORIGINAL_COLORS else None

    main_cell.write_svg(
        output_filename,
        scaling=scaling_factor,
        background=bg_color,
        style=style_spec
    )

    with open(output_filename, "r", encoding="utf-8") as file_handle:
        svg_string = file_handle.read()

    return svg_string


def main() -> None:
    """
    Main execution flow handling file upload, layout routing, policy application,
    inline display, and file downloading.
    """
    if ENABLE_GPU_WARMUP:
        warmup_gpu()

    print("Please upload your design file (.dxf or .oas):")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Execution terminated.")
        return

    for filename, file_content in uploaded.items():
        print(f"\nProcessing '{filename}'...")
        ext = os.path.splitext(filename)[1].lower()
        svg_output = ""

        # Using an ephemeral file prevents string decoding errors for complex CAD files
        temp_file = "ephemeral_upload" + ext
        with open(temp_file, "wb") as file_handle:
            file_handle.write(file_content)

        try:
            if ext == '.dxf':
                svg_output = process_dxf_to_svg(
                    temp_file,
                    DEFAULT_SVG_WIDTH_PX,
                    OUTPUT_SVG_FILENAME
                )
            elif ext in ['.oas', '.oasis']:
                svg_output = process_oasis_to_svg(
                    temp_file,
                    DEFAULT_SVG_WIDTH_PX,
                    OUTPUT_SVG_FILENAME
                )
            else:
                print(f"Unsupported file format: {ext}")
                os.remove(temp_file)
                continue

            # Render directly to the notebook cell output
            display_bg = "transparent" if USE_TRANSPARENT_BACKGROUND else ALTERNATIVE_BACKGROUND_COLOR
            # A subtle dashed border is added to the HTML container to define the canvas bounds if transparent
            html_container = f"<div style='width:{DEFAULT_SVG_WIDTH_PX}px; background-color:{display_bg}; border: 1px dashed #cccccc;'>{svg_output}</div>"
            display(HTML(html_container))

            print(f"Compilation complete. Downloading '{OUTPUT_SVG_FILENAME}'...")
            files.download(OUTPUT_SVG_FILENAME)

        except Exception as error:
            print(f"An error occurred while processing {filename}: {error}")
        finally:
            if os.path.exists(temp_file):
                os.remove(temp_file)

if __name__ == "__main__":
    main()

Initializing GPU context via CuPy...
GPU context initialized successfully.
Please upload your design file (.dxf or .oas):


Saving Goldy_with_Wafer.dxf to Goldy_with_Wafer (13).dxf

Processing 'Goldy_with_Wafer (13).dxf'...
Targeting Model Space directly. Found 79 root entities.
Flattening DXF block hierarchy (exploding nested inserts)...
Flattening geometry to Target Layer: '0'...


Compilation complete. Downloading 'flattened_layout.svg'...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>